In [1]:
import torch
import torch.nn.functional as F

from src.config import cfg
from src.tokenizer import Tokenizer

In [2]:
tokenizer = Tokenizer.load(cfg.paths.tokenizer)

ckpt = torch.load(
    f"{cfg.paths.checkpoint_dir}/latest.pt",
    map_location="cpu",
    weights_only=False,
)

state = ckpt["model"]

# u = state["u_embeddings.weight"]
emb = state["v_embeddings.weight"]

# emb = (u + v) / 2
emb = F.normalize(emb, dim=1)

In [3]:
def most_similar(word, topk=10):
    idx = tokenizer.index(word)

    sims = emb @ emb[idx]
    sims[idx] = -1e9

    ids = torch.topk(sims, topk).indices.tolist()

    return [(tokenizer.word(i), float(sims[i])) for i in ids]

In [4]:
# a - b + c
def analogy(a, b, c, topk=10):
    a_idx = tokenizer.index(a)
    b_idx = tokenizer.index(b)
    c_idx = tokenizer.index(c)

    query = emb[a_idx] - emb[b_idx] + emb[c_idx]
    query = F.normalize(query, dim=0)

    sims = emb @ query
    sims[[a_idx, b_idx, c_idx]] = -1e9

    ids = torch.topk(sims, topk).indices.tolist()

    return [(tokenizer.word(i), float(sims[i])) for i in ids]

In [5]:
analogies = [

    {
        "category": "cinsiyet",
        "expected": "kraliçe",
        "query": ("kral", "erkek", "kadın"),
    },
    {
        "category": "cinsiyet",
        "expected": "anne",
        "query": ("baba", "erkek", "kadın"),
    },
    {
        "category": "cinsiyet",
        "expected": "teyze",
        "query": ("amca", "erkek", "kadın"),
    },
    {
        "category": "cinsiyet",
        "expected": "kadın",
        "query": ("adam", "erkek", "kadın"),
    },
    {
        "category": "akrabalık",
        "expected": "kız",
        "query": ("oğul", "erkek", "kadın"),
    },

    {
        "category": "başkent",
        "expected": "paris",
        "query": ("ankara", "türkiye", "fransa"),
    },
    {
        "category": "başkent",
        "expected": "paris",
        "query": ("berlin", "almanya", "fransa"),
    },
    {
        "category": "başkent",
        "expected": "berlin",
        "query": ("londra", "ingiltere", "almanya"),
    },
    {
        "category": "başkent",
        "expected": "tokyo",
        "query": ("moskova", "rusya", "japonya"),
    },
    {
        "category": "başkent",
        "expected": "roma",
        "query": ("paris", "fransa", "italya"),
    },

    {
        "category": "şehir-ülke",
        "expected": "paris",
        "query": ("istanbul", "türkiye", "fransa"),
    },
    {
        "category": "şehir-ülke",
        "expected": "tokyo",
        "query": ("berlin", "almanya", "japonya"),
    },

    {
        "category": "fiil-zaman",
        "expected": "geldi",
        "query": ("gitmek", "gitti", "gelmek"),
    },
    {
        "category": "fiil-zaman",
        "expected": "okudu",
        "query": ("yazmak", "yazdı", "okumak"),
    },
    {
        "category": "fiil-zaman",
        "expected": "yürüdü",
        "query": ("koşmak", "koştu", "yürümek"),
    },

    {
        "category": "meslek-mekan",
        "expected": "öğretmen",
        "query": ("doktor", "hastane", "okul"),
    },
    {
        "category": "meslek-mekan",
        "expected": "doktor",
        "query": ("öğretmen", "okul", "hastane"),
    },
    {
        "category": "araç",
        "expected": "kaptan",
        "query": ("pilot", "uçak", "gemi"),
    },

    {
        "category": "ülke-yönetim",
        "expected": "başkan",
        "query": ("cumhurbaşkanı", "türkiye", "amerika"),
    },

    {
        "category": "akrabalık",
        "expected": "baba",
        "query": ("anne", "kadın", "erkek"),
    },
    {
        "category": "akrabalık",
        "expected": "abi",
        "query": ("abla", "kadın", "erkek"),
    },

    {
        "category": "kıta-ülke",
        "expected": "avrupa",
        "query": ("asya", "çin", "fransa"),
    },

    {
        "category": "spor",
        "expected": "raket",
        "query": ("futbol", "top", "tenis"),
    },
    {
        "category": "spor",
        "expected": "kale",
        "query": ("basketbol", "pota", "futbol"),
    },

    {
        "category": "teknoloji",
        "expected": "sql",
        "query": ("python", "programlama", "veritabanı"),
    },
]

In [6]:
for item in analogies:

    a, b, c = item["query"]

    print(f"\n[{item['category']}]")
    print(f"Beklenen: {item['expected']}")
    print(f"{a} - {b} + {c}")

    print(analogy(a, b, c))


[cinsiyet]
Beklenen: kraliçe
kral - erkek + kadın
[('kralı', 0.6925574541091919), ('kralın', 0.6377193927764893), ('kraliçe', 0.6340086460113525), ('kralının', 0.6098556518554688), ('krallığı', 0.6073781251907349), ('prens', 0.6064103841781616), ('hükümdar', 0.6062918305397034), ('kraliyet', 0.5916690230369568), ('krala', 0.5897341370582581), ('hükümdarı', 0.5889414548873901)]

[cinsiyet]
Beklenen: anne
baba - erkek + kadın
[('babanın', 0.6222230792045593), ('anne', 0.6052874326705933), ('oğul', 0.5804584622383118), ('harabati', 0.5608288049697876), ('babaya', 0.5547136068344116), ('kutbûd', 0.5535181760787964), ('બ', 0.5505685806274414), ('špela', 0.5486190319061279), ('şücâed', 0.5468964576721191), ('ağuçan', 0.5467960238456726)]

[cinsiyet]
Beklenen: teyze
amca - erkek + kadın
[('cumali', 0.6059742569923401), ('aliçoyu', 0.596296489238739), ('amcanın', 0.5954397916793823), ('aliçonun', 0.5940853357315063), ('erdalı', 0.5936462879180908), ('şukufeyi', 0.5922912955284119), ('kutmanın